## Step 1: Install

In [ ]:
!pip install -q pdfplumber sentence-transformers pandas tqdm "pillow>=8.0,<12.0"

## Step 2: Imports

In [ ]:
import pdfplumber
import pandas as pd
import numpy as np
import re, gc, os, zipfile
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from google.colab import files

## Step 3: Unzip reports

Drag and drop your 3 zip files into  using the Colab sidebar, then run this cell.

In [ ]:
from google.colab import files
import zipfile, os
from pathlib import Path

os.makedirs("/content/reports", exist_ok=True)

for label in ["AIMS_Reports_GBR", "MMP_Water_Quality", "Reef_Updates_Reports"]:
    print(f"Upload {label}.zip:")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        zip_path = f"/content/{fname}"
        with open(zip_path, "wb") as f:
            f.write(data)
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall("/content/reports")
            print(f"Extracted: {fname}")
        except zipfile.BadZipFile:
            print(f"BAD ZIP: {fname}")

AIMS_FOLDER = "/content/reports/AIMS Reports GBR"
MMP_FOLDER  = "/content/reports/MMP_Water Quality"
REEF_FOLDER = "/content/reports/Reef Updates Reports"

for name, folder in [("AIMS", AIMS_FOLDER), ("MMP", MMP_FOLDER), ("Reef", REEF_FOLDER)]:
    count = len(list(Path(folder).rglob("*.pdf")))
    print(f"{name}: {count} PDFs found")

## Step 4: Date extractor

In [ ]:
MONTHS = {
    "january":"01","february":"02","march":"03","april":"04",
    "may":"05","june":"06","july":"07","august":"08",
    "september":"09","october":"10","november":"11","december":"12"
}
VALID_YEARS = [str(y) for y in range(2018, 2026)]

def extract_period_range(text, filename):
    """
    Returns (start_period, end_period) as YYYY-MM strings.
    Handles 3 cases:
      1. Explicit month+year  -> single month (start==end)
      2. Year range (2018-19, 2018/2019, 2018_2019) -> full Jul-Jun range
      3. Single year only      -> full Jan-Dec of that year
    """
    t = (filename + " " + text[:6000]).lower()

    # Case 1: explicit month + year (e.g. "April 2025")
    hit = re.search(
        r'(?:\d{1,2}\s+)?(january|february|march|april|may|june|july|'
        r'august|september|october|november|december)\s+(20\d{2})', t
    )
    if hit:
        p = f"{hit.group(2)}-{MONTHS[hit.group(1)]}"
        return p, p

    # Case 2: year range like 2018-19, 2018_2019, 2018/19
    hit = re.search(r'(20\d{2})[-_/](\d{2,4})', t)
    if hit:
        y1 = int(hit.group(1))
        y2_raw = hit.group(2)
        y2 = int(y2_raw) if len(y2_raw) == 4 else int(str(y1)[:2] + y2_raw)
        if y2 == y1 + 1:
            # Aussie reporting year often runs Jul Y1 -> Jun Y2
            return f"{y1}-07", f"{y2}-06"
        elif y2 > y1:
            return f"{y1}-01", f"{y2}-12"

    # Case 3: single plain year -> spans the whole year
    for y in reversed(VALID_YEARS):
        if y in t:
            return f"{y}-01", f"{y}-12"

    return None, None


## Step 5: Region text extractor

In [ ]:
NOISE_PATTERNS = [
    # Abbreviations / boilerplate
    "abbreviations", "acronyms", "bureau of meteorology",
    "institute of marine science", "james cook university",
    "tropwater", "townsville mc", "po box", "issn",
    "creative commons", "traditional sea country", "traditional owners",
    "yadhaykenu", "wuthathi", "copyright", "photographer",
    "comments and questions", "this project is supported",
    "front cover", "back cover", "cite as", "doi.org",
    # Acknowledgements
    "we thank", "we are grateful", "we would like to thank",
    "the authors thank", "the program thanks",
    "lama lama rangers", "yuku baja muliku",
    "yintinga aboriginal corporation", "rinyirru aboriginal corporation",
    "frontier fishing charters", "blue planet marine",
    "mission beach charters", "barra charters",
    "eric dick", "jeff shellberg", "sarah herkess",
    "field work", "fieldwork", "involved in the field",
    "sampling water quality and tracking",
    # URLs / references
    "www.reef", "www.gbr", "reefauthority.gov",
    "saved from", "csiro commonwealth"
]

def is_noise_sentence(s):
    s_lower = s.lower()
    return any(p in s_lower for p in NOISE_PATTERNS)

def is_toc_line(sentence):
    dot_ratio = sentence.count('.') / max(len(sentence), 1)
    has_page_ref = bool(re.search(r'\.{3,}\s*\d+', sentence))
    is_short_header = len(sentence.split()) < 6 and re.search(r'\d', sentence)
    return dot_ratio > 0.2 or has_page_ref or bool(is_short_header)

def is_garbled(text):
    """Detect if PDF text extraction is fundamentally broken."""
    words = text.split()
    if not words: return True
    avg_len = sum(len(w) for w in words) / len(words)
    # If average word length > 15, words are merged — unfixable
    return avg_len > 15

def fix_garbled(text):
    """Attempt to fix mild garbling — insert spaces before capitals/digits."""
    words = text.split()
    avg_len = sum(len(w) for w in words) / max(len(words), 1)
    if 10 < avg_len <= 15:
        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
        text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
        text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)
        text = re.sub(r'\s+', ' ', text)
    return text

def get_region_text(text, region):
    text = fix_garbled(text)
    t    = text.lower()

    keys = (
        ["northern region", "far northern", "lizard island",
         "port douglas", "cooktown", "cape york", "north gbr",
         "northern gbr", "wet tropics", "northern great barrier"]
        if region == "North" else
        ["central region", "townsville", "whitsunday", "mackay",
         "burdekin", "central gbr", "central great barrier",
         "proserpine", "bowen"]
    )

    sents = re.split(r'(?<=[.!?])\s+', t)

    keep = []
    for s in sents:
        s = s.strip()
        if len(s.split()) < 6:
            continue
        if is_toc_line(s):
            continue
        if is_noise_sentence(s):
            continue
        if any(k in s for k in keys):
            keep.append(s)

    if not keep:
        keep = [s.strip() for s in sents
                if not is_toc_line(s)
                and not is_noise_sentence(s)
                and len(s.split()) >= 6][:30]

    return " ".join(keep)

## Step 6: Read all PDFs from all three folders

In [ ]:
# PDFs with known broken encoding — skip entirely
SKIP_PDFS = {
    "AIMS_LTMP_Report_on GBR_coral_status_2021_2022_040822F3.pdf"
}

def is_toc_page(text):
    if not text: return False
    dot_ratio    = text.count('.') / max(len(text), 1)
    lines        = text.split('\n')
    dotted_lines = sum(1 for l in lines if '...' in l or re.search(r'\.{3,}\s*\d+', l))
    return dot_ratio > 0.15 or (dotted_lines / max(len(lines), 1)) > 0.25

def is_abbrev_page(text):
    if not text: return False
    t = text.lower()
    abbrev_hits = sum(1 for p in [
        "abbreviations", "acronyms", "bureau of meteorology",
        "aims australian", "bom bureau", "cdom colour", "commonly used"
    ] if p in t)
    return abbrev_hits >= 2

def is_garbled_page(text):
    if not text: return False
    words = text.split()
    if len(words) < 10: return False
    avg_len = sum(len(w) for w in words) / len(words)
    return avg_len > 15

def build_df(folder, label):
    rows = []
    pdfs = list(Path(folder).rglob("*.pdf")) + list(Path(folder).rglob("*.PDF"))
    print(f"[{label}] PDFs found: {len(pdfs)}")
    for pdf in tqdm(pdfs, desc=label):

        # Skip known garbled PDFs
        if pdf.name in SKIP_PDFS:
            print(f"  Skipping (garbled encoding): {pdf.name}")
            continue

        txt         = ""
        period_text = ""
        try:
            with pdfplumber.open(pdf) as f:
                for i, page in enumerate(f.pages):
                    if i >= 60: break
                    p = page.extract_text()
                    if not p: continue
                    if i < 10:
                        period_text += " " + p
                    if i == 0: continue
                    if is_abbrev_page(p): continue
                    if is_toc_page(p): continue
                    if is_garbled_page(p): continue   # skip garbled pages
                    txt += " " + p
        except:
            continue

        txt = re.sub(r"\s+", " ", txt)
        if len(txt.strip()) < 100:
            continue

        period_text = re.sub(r"\s+", " ", period_text)
        start_period, end_period = extract_period_range(period_text, pdf.name)
        if start_period is None:
            continue

        for region in ["North", "Central"]:
            rows.append({
                "file":         pdf.name,
                "start_period": start_period,
                "end_period":   end_period,
                "region":       region,
                "text":         get_region_text(txt, region)
            })
        gc.collect()
    return pd.DataFrame(rows)

aims_df = build_df(AIMS_FOLDER, "AIMS")
mmp_df  = build_df(MMP_FOLDER,  "MMP")
reef_df = build_df(REEF_FOLDER, "Reef")

all_df = pd.concat([aims_df, mmp_df, reef_df], ignore_index=True)

print(f"\nTotal reports parsed: {len(all_df)}")
for _, r in all_df.drop_duplicates("file").iterrows():
    print(f"  {r['file']}: {r['start_period']} -> {r['end_period']}")

## Step 6b: Diagnostic — verify extracted text quality

Check what sentences were actually pulled for each PDF and region before embedding.

In [ ]:
print("=" * 70)
print("EXTRACTED TEXT QUALITY CHECK")
print("=" * 70)

for _, row in all_df.drop_duplicates(["file", "region"]).iterrows():
    print(f"\nFile   : {row['file']}")
    print(f"Period : {row['start_period']} -> {row['end_period']}")
    print(f"Region : {row['region']}")
    print(f"Text length: {len(row['text'])} chars  |  Words: {len(row['text'].split())}")
    print(f"Preview (first 400 chars):")
    print("-" * 40)
    print(row['text'][:400])
    print("-" * 40)

# Summary stats
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Total rows: {len(all_df)}")
print(f"Unique files: {all_df['file'].nunique()}")
print(f"\nText length stats (words):")
all_df["word_count"] = all_df["text"].str.split().str.len()
print(all_df.groupby(["file","region"])["word_count"].sum().reset_index().to_string())

# Flag suspiciously short extractions
short = all_df[all_df["word_count"] < 50]
if len(short) > 0:
    print(f"\n⚠️  Rows with < 50 words (may be poor extraction):")
    print(short[["file","region","word_count","text"]].to_string())
else:
    print("\n✅ All rows have >= 50 words")

## Step 7: Build full 2018–2025 calendar and combine texts

In [ ]:
# ── Explode each report across every month it covers ──────
months_idx = pd.date_range(start="2018-01-01", end="2025-12-01", freq="MS").strftime("%Y-%m")
month_to_idx = {m: i for i, m in enumerate(months_idx)}

exploded_rows = []
for _, r in all_df.iterrows():
    if r["start_period"] not in month_to_idx or r["end_period"] not in month_to_idx:
        continue
    start_i = month_to_idx[r["start_period"]]
    end_i   = month_to_idx[r["end_period"]]
    for i in range(start_i, end_i + 1):
        exploded_rows.append({
            "period": months_idx[i],
            "region": r["region"],
            "text":   r["text"]
        })

exploded_df = pd.DataFrame(exploded_rows)
print(f"Exploded rows: {len(exploded_df)}")

# Aggregate (multiple reports can cover the same period)
agg = (
    exploded_df.groupby(["period","region"])["text"]
    .apply(lambda x: " ".join(x.dropna()))
    .reset_index()
)

# Full calendar: 192 rows (12 months x 8 years x 2 regions)
calendar = pd.DataFrame({"period": months_idx})
calendar = calendar.merge(pd.DataFrame({"region": ["North","Central"]}), how="cross")

monthly = calendar.merge(agg, on=["period","region"], how="left")
monthly["text"] = monthly["text"].fillna("").str.strip()

print("Calendar rows:", len(monthly), " (should be 192)")
print("Rows with real text:", (monthly["text"].str.len() > 0).sum())
print("Rows still empty   :", (monthly["text"].str.len() == 0).sum())

## Step 8: Chunked mean pooling embeddings

Splits each text into ~200-word chunks → embeds each with MiniLM → averages → one 384-dim vector per (period, region).

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

CHUNK_WORDS = 200

def chunk_text(text, chunk_size=CHUNK_WORDS):
    words = text.split()
    if not words:
        return ["no data available"]
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def embed_text(text):
    chunks = chunk_text(text)
    vecs   = model.encode(chunks, batch_size=16, show_progress_bar=False)
    return vecs.mean(axis=0)   # (384,)

print(f"Embedding {len(monthly)} rows ...")
embeddings = np.vstack([embed_text(t) for t in tqdm(monthly["text"])])
print("Shape:", embeddings.shape)  # (192, 384)

## Step 9: Assemble and download CSV

In [ ]:
emb_df = pd.DataFrame(embeddings, columns=[f"emb_{i+1}" for i in range(384)])

reef_final = pd.concat(
    [monthly[["period","region"]].reset_index(drop=True), emb_df],
    axis=1
).sort_values(["period","region"]).reset_index(drop=True)

print("Final shape:", reef_final.shape)   # (192, 386)
print("Years:", sorted(reef_final["period"].str[:4].unique()))

OUTPUT_PATH = "/content/reef_minilm_embeddings_2018_2025.csv"
reef_final.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

files.download(OUTPUT_PATH)
print("Download started.")